# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M5 — LangGraph-Style Stateful Care Coordination Workflow

### Goal
Turn the M1 parser + M2 tools + M4 retrieval into a stateful workflow.

The graph separates operational stages rather than letting one unrestricted prompt do everything.

In [5]:
# Verify the selected notebook kernel before installing anything.
import sys

if sys.version_info[:2] != (3, 14) or sys.prefix == sys.base_prefix:
    raise RuntimeError(
        "Select the Python 3.14 (.venv314) notebook kernel and restart it before running setup. "
        f"Current interpreter: {sys.executable}"
    )

print(f"Using Python {sys.version.split()[0]} from {sys.executable}")

# Pinned for this cohort - do not un-pin, so a framework update can't break the lab mid-session.
# langgraph                     : the graph runtime
# langgraph-checkpoint-sqlite   : the DURABLE checkpointer - a SEPARATE package, required by Part B7
# langchain-litellm             : ChatLiteLLM - LiteLLM (the program-standard client)
#                                 wrapped as a LangChain chat model, so it plugs straight into
#                                 LangGraph: .bind_tools(), message objects, add_messages reducer.
# litellm / python-dotenv       : the underlying client + the .env loader
# (graph pictures need no extra package - .draw_mermaid_png() ships with langgraph)
# (No -q: if an install fails you want the full pip error visible, not silenced.)
%pip install "langgraph>=1.2,<2.0" "langgraph-checkpoint-sqlite>=3.0,<4.0" "python-dotenv>=1.0"
%pip install "litellm>=1.93,<2.0" "langchain-litellm>=0.7,<0.8"

print("Dependencies installed. If pip asks you to restart the kernel, do so, then continue.")

# """
# EXPECTED OUTPUT
# ---------------
# Using Python 3.14.x from .../.venv314/bin/python
# (pip install log)
# Dependencies installed. If pip asks you to restart the kernel, do so, then continue.
# """

Using Python 3.14.7 from /Users/sgunaseelan/Projects/API/AI-IIT/applied-ai-capstone-starter/.venv-py314/bin/python
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Dependencies installed. If pip asks you to restart the kernel, do so, then continue.


In [6]:
from dataclasses import dataclass, field
from typing import Any, Callable
from IPython.display import display, SVG
import graphviz

@dataclass
class State:
    raw_message: str
    patient_ref: str | None = None
    intake: dict[str, Any] = field(default_factory=dict)
    tool_result: Any = None
    retrieved: list[dict] = field(default_factory=list)
    route: str = "unknown"
    answer: str = ""
    events: list[str] = field(default_factory=list)

# ------------------------------------------------------------------
# M5 workflow graph
# ------------------------------------------------------------------
# We render this with Graphviz instead of emitting Mermaid source as text.
# This works directly in Jupyter/VS Code when the graphviz Python package
# and Graphviz system executable are installed.

def build_workflow_graph():
    dot = graphviz.Digraph("CareFlow_M5", format="svg")
    dot.attr(rankdir="TB", bgcolor="transparent", pad="0.25", nodesep="0.35", ranksep="0.55")
    dot.attr("node", shape="box", style="rounded,filled", fontname="Helvetica", fontsize="11", margin="0.15,0.08")
    dot.attr("edge", fontname="Helvetica", fontsize="9")

    dot.node("start", "START", shape="circle", style="filled")
    dot.node("parse_intake", "parse_intake")
    dot.node("clinical_refusal", "clinical_refusal")
    dot.node("retrieve_policy", "retrieve_policy")
    dot.node("policy_not_found", "policy_not_found")
    dot.node("respond_policy", "respond_policy")
    dot.node("tool_lookup", "tool_lookup")
    dot.node("respond_tool", "respond_tool")
    dot.node("end", "END", shape="doublecircle", style="filled")

    dot.edge("start", "parse_intake")
    dot.edge("parse_intake", "clinical_refusal", label="clinical advice")
    dot.edge("parse_intake", "retrieve_policy", label="policy question")
    dot.edge("parse_intake", "tool_lookup", label="operations")
    dot.edge("retrieve_policy", "policy_not_found", label="no evidence")
    dot.edge("retrieve_policy", "respond_policy", label="evidence found")
    dot.edge("tool_lookup", "respond_tool")
    dot.edge("clinical_refusal", "end")
    dot.edge("policy_not_found", "end")
    dot.edge("respond_policy", "end")
    dot.edge("respond_tool", "end")
    return dot

workflow_graph = build_workflow_graph()
svg_bytes = workflow_graph.pipe(format="svg")
display(SVG(svg_bytes.decode("utf-8")))


ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
# ------------------------------------------------------------------
# Stateful workflow implementation
# ------------------------------------------------------------------

def parse_intake(state: State) -> State:
    state.events.append("parse_intake")
    q = state.raw_message.lower()
    clinical_terms = ["should i", "dosage", "dose", "medication", "interpret my symptoms", "what does this symptom mean"]
    if any(term in q for term in clinical_terms):
        state.route = "clinical_refusal"
    elif any(term in q for term in ["appointment", "referral", "eligibility", "insurance", "plan"]):
        state.route = "tool_lookup" if any(term in q for term in ["appointment", "referral", "eligibility"]) else "retrieve_policy"
    else:
        state.route = "retrieve_policy"
    return state

def clinical_refusal(state: State) -> State:
    state.events.append("clinical_refusal")
    state.answer = (
        "This request requires a licensed clinician. CareFlow will not interpret "
        "symptoms or provide medication/dosing advice."
    )
    return state

def retrieve_policy(state: State) -> State:
    state.events.append("retrieve_policy")
    # Replace this with the M4 retriever in the integrated project.
    state.retrieved = [{"source": "demo-policy", "evidence": True}] if "copay" in state.raw_message.lower() else []
    state.route = "respond_policy" if state.retrieved else "policy_not_found"
    return state

def policy_not_found(state: State) -> State:
    state.events.append("policy_not_found")
    state.answer = "The available CareFlow policy corpus does not cover that question."
    return state

def respond_policy(state: State) -> State:
    state.events.append("respond_policy")
    state.answer = "Answer using only the retrieved policy evidence and cite the source document."
    return state

def tool_lookup(state: State) -> State:
    state.events.append("tool_lookup")
    # Replace with M2/M6 tool calls in the integrated project.
    state.tool_result = {"status": "tool_called", "patient_ref": state.patient_ref}
    return state

def respond_tool(state: State) -> State:
    state.events.append("respond_tool")
    state.answer = "Return the deterministic tool result without inventing operational state."
    return state

def run_workflow(state: State) -> State:
    state = parse_intake(state)
    if state.route == "clinical_refusal":
        return clinical_refusal(state)
    if state.route == "retrieve_policy":
        state = retrieve_policy(state)
        return respond_policy(state) if state.route == "respond_policy" else policy_not_found(state)
    state = tool_lookup(state)
    return respond_tool(state)

# Compact demonstration: do NOT print the full tool/database payload.
examples = [
    State(raw_message="Should I take another tablet for this symptom?", patient_ref="MHP-P-00001"),
    State(raw_message="What is the specialist copay?", patient_ref="MHP-P-00001"),
    State(raw_message="Show my next appointment", patient_ref="MHP-P-00001"),
]

for example in examples:
    result = run_workflow(example)
    print({
        "route": result.route,
        "events": result.events,
        "answer": result.answer,
    })


### Why state matters

The state object makes routing decisions explicit and inspectable:

```text
START
  │
  ▼
parse_intake
  ├── clinical advice ─────► clinical_refusal ─────► END
  ├── policy question ─────► retrieve_policy
  │                            ├── evidence ───────► respond_policy ─► END
  │                            └── no evidence ────► policy_not_found ─► END
  └── operations ───────────► tool_lookup ─────────► respond_tool ───► END
```

### Integration boundary

- **M1** supplies structured intake/routing signals.
- **M2** supplies deterministic operational tools.
- **M4** supplies policy retrieval.
- **M5** orchestrates these capabilities as an explicit state machine.
- **M6** can replace local tool calls with MCP-backed tools and introduce multiple agents.

### Why Graphviz instead of raw Mermaid

The previous cell emitted Mermaid syntax as plain text. Jupyter/VS Code does not automatically render arbitrary Mermaid source in a normal Python output cell. This notebook now creates an SVG graph directly, so the workflow is visible in the notebook itself.
